# 02 · Uncertainty fingerprints (≈ 12 min)
**Challenge 3: see how different models have different fingerprints.**

**No code to write:** run the two cells, read the grids, compare.

A **fingerprint** is the box an answer lands in. For each question we look at two signals from notebook 01:
**SC** (do the 10 answers use the same words?) and **SE** (do they mean the same thing?).
We cut each at its **median** (the middle value), which gives four boxes:

| Box | SC | SE | Reading |
|---|---|---|---|
| Confident | low | low | the 10 answers agree |
| Gap-like | high | high | the 10 answers disagree |
| Shallow | low | high | same words, different meanings (rare) |
| Surface-varying | high | low | different words, same meaning (rare) |

No answer key is used to build the boxes, so this could run in a real system.

In [ ]:
import sys, pathlib, warnings
from collections import Counter
REPO = "pycon2026-uncertainty-workshop"   # <- must match your GitHub repo name (used on Colab only)
if "google.colab" in sys.modules:
    ROOT = pathlib.Path("/content") / REPO
    if not ROOT.exists():
        import subprocess
        subprocess.run(["git", "clone", f"https://github.com/Shafiya0101/{REPO}.git", str(ROOT)], check=True)
else:
    ROOT = pathlib.Path.cwd()
    if ROOT.name in ("notebooks", "solutions"):
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from uncertainty_lib import *
from uncertainty_lib import checks
from uncertainty_lib.plotting import set_style, plot_region_grid, plot_risk_coverage
set_style()
pd.set_option("display.max_colwidth", 70)

## Step 1 · Run: the fingerprint of Mistral-7B
**Look for:** in which box are most of the wrong answers? How many questions are in each box?

In [ ]:
def show_fingerprint(setting):
    df = add_signals(load_setting(setting))
    checks.banner(df)
    tau_sc, tau_se = fit_medians(df)
    df = add_regions(df, tau_sc, tau_se)
    summary = region_summary(df)
    print(f"{setting}: medians  SC = {tau_sc:.2f}   SE = {tau_se:.2f}")
    print(summary.round(3))
    plot_region_grid(summary, title=f"Wrong answers per box: {setting}")
    plt.show()

show_fingerprint("mistral_triviaqa")

## Step 2 · Your turn: change the model
In the cell below, change the name to **`"qwen2_1_5b_triviaqa"`** (a smaller model) and run it.
You can also try `"qwen2_7b_triviaqa"`, `"mistral_nq"` or `"mistral_hotpotqa"`.

**Look for:** did the medians move? How many answers are wrong in the *Confident* box now?

In [ ]:
show_fingerprint("mistral_triviaqa")      # <- change this name, then run

## What to take away
**Different models have different fingerprints.**
* On the real data, Mistral-7B's *Confident* box is mostly right (about 6.5% wrong), while its *Gap-like* box is wrong about 42% of the time.
* For the smaller Qwen2-1.5B, the medians move a lot (SC from 0.20 to 0.70), and its *Confident* box is wrong about 40% of the time.
* So "Confident" is only **relative to the model**: the cut-offs must be fitted for each model and each dataset, and a confident answer is not always a correct one.

*Optional, for later:* `unanimous_errors(df)` lists answers where all 10 samples agree and the answer is still wrong (the "confident fabrication").